# BLEURT Correctness Threshold Sweep Analysis ($\tau \in [-0.70, +0.50]$)

This notebook evaluates the operational stability and ranking preservation of uncertainty quantification (UQ) methods across diverse correctness thresholds $\tau \in [-0.70, +0.50]$ on multiple datasets (**TruthfulQA**, **ScienceQA**, etc.).

### Contenders Evaluated:
1. **Difference Uncertainty Decomposition (Ours):**
   - Aleatoric Uncertainty ($AU$): $H_{\text{mix}}^{(K_a^*)}$ (Parameter Size-Class Optimal Horizon)
   - Epistemic Uncertainty ($EU$): $\Delta H = H_{\text{final}} - H_{\text{mix}}^{(K_e^*)}$
   - Total Uncertainty ($TU$): $H_{\text{final}}$ / $H_{\text{mix}}^{(K_a^*)} + \Delta H$
2. **Asymmetric Decomposition (Ours):** Gated Mutual Information ($EU$), Premature Mixture Entropy ($AU$)
3. **Single-Layer Baseline:** Single-Layer Final Entropy $H_{\text{final}}$
4. **External Decomposed Baselines:** TokUR ($AU$, $EU$, $\text{Total}$), ICE ($AU$, $EU$, $\text{Total}$)
5. **Semantic Entropy Baselines:** Our Weighted SE, SDLG SE Baseline (Unweighted) [Aichberger et al., 2025], Kuhn SE Baseline


In [ ]:
%matplotlib inline
import os
import glob
import pickle as pkl
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, roc_curve
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

GLOBAL_CORR_THRESHOLD = 0.0
EVAL_LABEL_KEY = 'bleurt'

sns.set_theme(style="whitegrid", font="sans-serif")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 1.2


## 1. Candidate Directory Discovery & Dynamic Hybrid Baseline Loader
Dynamically discovers run files across candidate result directories without hardcoded sample limits. Accommodates 3-chunk structures (`Ours`, `SDLG`, `Kuhn`) using dynamic chunk-stride resolution.

In [ ]:
# Configuration - Candidate directories (Unified prioritized, followed by entropies3 and entropies2)
CANDIDATES = [
    '../experiments/result/entropies_unified',
    'src/experiments/result/entropies_unified',
    './result/entropies_unified',
    '../../result/entropies_unified',
    'result/entropies_unified',
    'entropies_unified',
    '../experiments/result/entropies3',
    '../experiments/results/entropies3',
    'src/experiments/result/entropies3',
    'src/experiments/results/entropies3',
    './result/entropies3',
    '../../result/entropies3',
    'result/entropies3',
    'entropies3',
    '../experiments/result/entropies2',
    '../experiments/results/entropies2',
    'src/experiments/result/entropies2',
    'src/experiments/results/entropies2',
    './result/entropies2',
    '../../result/entropies2',
    'result/entropies2',
    'entropies2',
    'src/temp',
    './src/temp',
    '../src/temp',
    'src',
    './src',
    '.'
]

valid_dirs = []
for c in CANDIDATES:
    if os.path.exists(c) and any(f.endswith('.pkl') for f in os.listdir(c)):
        abs_c = os.path.abspath(c)
        if abs_c not in valid_dirs:
            valid_dirs.append(abs_c)

DATA_DIR = valid_dirs[0] if valid_dirs else '.'
print(f"Primary Data Directory: {DATA_DIR}")
print(f"All Valid Candidate Directories: {valid_dirs}")

GLOBAL_CORR_THRESHOLD = 0.0
EVAL_LABEL_KEY = 'bleurt'

# Discover all run files dynamically across valid directories in priority order (entropies_unified > entropies3 > entropies2)
runs = []
seen_run_keys = set()

for d_path in valid_dirs:
    d_files = glob.glob(os.path.join(d_path, "*.pkl"))
    entropy_files = [f for f in d_files if "_entropies_" in os.path.basename(f) or os.path.basename(f).startswith("sdlg_entropies_")]
    for f in sorted(entropy_files):
        base = os.path.basename(f)
        label_file = f.replace("_entropies_", "_label_").replace("sdlg_entropies_", "sdlg_label_")
        if not os.path.exists(label_file):
            if 'qwen-1.5b' in label_file: label_file = label_file.replace('qwen-1.5b', 'qwen2.5-1.5b')
            elif 'qwen2.5-1.5b' in label_file: label_file = label_file.replace('qwen2.5-1.5b', 'qwen-1.5b')
        question_file = f.replace("_entropies_", "_question_").replace("sdlg_entropies_", "sdlg_question_")
        parts = base.replace(".pkl", "").split("_")
        model_name = "unknown"
        dataset_name = "truthfulqa"
        for p in parts:
            p_lower = p.lower()
            if any(m in p_lower for m in ['llama', 'qwen', 'phi', 'opt', 'mistral', 'gemma', 'deepseek']): model_name = p
            if any(d in p_lower for d in ['truth', 'sciq', 'openbook', 'open', 'science', 'scienceqa', 'ambig']):
                dataset_name = 'scienceqa' if 'science' in p_lower or 'sciq' in p_lower else ('openbookqa' if 'open' in p_lower else ('truthfulqa' if 'truth' in p_lower else p_lower))
                
        if os.path.exists(label_file):
            run_key = (dataset_name.lower(), model_name.lower())
            if run_key not in seen_run_keys:
                seen_run_keys.add(run_key)
                runs.append({
                    'name': f"{model_name} on {dataset_name}",
                    'model': model_name,
                    'dataset': dataset_name,
                    'entropy_file': f,
                    'label_file': label_file,
                    'question_file': question_file if os.path.exists(question_file) else None,
                    'dir': d_path
                })

print(f"Discovered {len(runs)} valid evaluation runs across datasets.")
for r in runs:
    print(f"  - [{r['dataset']}] {r['model']} (from {os.path.basename(r['dir'])})")


## 1.5 Run Dataframe Loader & Model Horizon Specifications
Configures model capacity horizons, metric candidate resolvers, and dynamic multi-chunk loader for all uncertainty baselines (TokUR, ICE, Semantic Entropy, Divergences, Single-Layer, and Difference Decomposition).

In [ ]:
MODEL_CLASS_HORIZONS = {
    'large': {'class': 'Large (14B+)', 'Ka': 3, 'Ke': 32},
    'mid':   {'class': 'Mid (7B-8B)',   'Ka': 12, 'Ke': 28},
    'small': {'class': 'Small (1.5B-3B)', 'Ka': 2, 'Ke': 27}
}

def get_model_class_and_horizons(model_name):
    m = str(model_name).lower().replace('_', '-')
    if any(s in m for s in ['14b', 'phi-3-medium']):
        return 'Large (14B+)', 3, 32
    elif any(s in m for s in ['7b', '8b']):
        return 'Mid (7B-8B)', 12, 28
    else:
        return 'Small (1.5B-3B)', 2, 27

def safe_extract_label(val, key='bleurt'):
    while isinstance(val, (list, tuple)) and len(val) > 0: val = val[0]
    if isinstance(val, dict):
        if key in val: val = val[key]
        elif 'correctness_dict' in val: val = val['correctness_dict'].get(key, 0.0)
        else:
            for k in ['bleurt_score', 'bleurt', 'rougeL', 'rouge1', 'exact_match']:
                if k in val: val = val[k]; break
            else: val = 0.0
    try:
        fval = float(val)
        return fval if np.isfinite(fval) else 0.0
    except Exception:
        return 0.0

def _to_fixed_array(chunk_or_list, key, N):
    if N <= 0: return np.array([], dtype=float)
    if chunk_or_list is None: return np.full(N, np.nan, dtype=float)
    if isinstance(chunk_or_list, list):
        if key is not None:
            raw = [d.get(key, np.nan) if isinstance(d, dict) else np.nan for d in chunk_or_list[:N]]
        else:
            raw = chunk_or_list[:N]
    else:
        raw = list(chunk_or_list)[:N]
    arr = pd.to_numeric(pd.Series(raw), errors='coerce').values
    if len(arr) < N:
        arr = np.pad(arr, (0, N - len(arr)), constant_values=np.nan)
    elif len(arr) > N:
        arr = arr[:N]
    return arr.astype(float)

def _model_aliases(model):
    m = str(model).lower().replace('_', '-').replace('-instruct', '').replace('-chat', '').strip('-')
    aliases = [m, m.replace('-', '_'), m.replace('-', '')]
    if 'meta-' in m:
        aliases.extend([m.replace('meta-', ''), m.replace('meta-', '').replace('-', '')])
    else:
        aliases.append(f"meta-{m}")

    if 'qwen-1.5b' in m or 'qwen2.5-1.5b' in m:
        aliases.extend(['qwen-1.5b', 'qwen2.5-1.5b', 'qwen_1.5b', 'qwen2.5_1.5b', 'qwen2.5-1.5b-instruct', 'qwen1.5b', 'qwen2.5-1.5'])
    if 'qwen-3b' in m or 'qwen2.5-3b' in m:
        aliases.extend(['qwen-3b', 'qwen2.5-3b', 'qwen_3b', 'qwen2.5_3b', 'qwen2.5-3b-instruct', 'qwen3b', 'qwen2.5-3'])
    if 'qwen-7b' in m or 'qwen2.5-7b' in m:
        aliases.extend(['qwen-7b', 'qwen2.5-7b', 'qwen_7b', 'qwen2.5_7b', 'qwen2.5-7b-instruct', 'qwen7b', 'qwen2.5-7'])
    if 'qwen-14b' in m or 'qwen2.5-14b' in m:
        aliases.extend(['qwen-14b', 'qwen2.5-14b', 'qwen_14b', 'qwen2.5_14b', 'qwen2.5-14b-instruct', 'qwen14b', 'qwen2.5-14'])
    if 'qwen1.5-moe' in m or 'qwen-moe' in m or 'moe' in m:
        aliases.extend(['qwen1.5-moe-a2.7b', 'qwen1.5_moe_a2.7b', 'qwen1.5-moe', 'qwen_moe', 'qwen1.5-moe-chat', 'moe-a2.7b'])
    if 'llama-3.2-3b' in m or 'llama3.2-3b' in m:
        aliases.extend(['llama-3.2-3b', 'llama3.2-3b', 'llama-3-3b', 'llama-3.2-3b-instruct', 'llama3.2_3b', 'llama-3.2', 'llama3.2'])
    if 'llama-3-8b' in m or 'meta-llama-3-8b' in m or 'llama3-8b' in m or 'llama-8b' in m or 'llama8b' in m:
        aliases.extend(['llama-3-8b', 'llama3-8b', 'meta-llama-3-8b', 'meta-llama-3-8b-instruct', 'llama-3-8b-instruct', 'llama-8b', 'llama8b', 'meta-llama-3.1-8b', 'llama-3.1-8b', 'llama3.1-8b', 'meta-llama-3', 'llama3'])
    if 'phi-3-mini' in m or 'phi-3-mini-4k' in m or 'phi-3' in m:
        aliases.extend(['phi-3-mini-4k', 'phi-3-mini', 'phi-3', 'phi-3-mini-4k-instruct', 'phi3'])
    if 'phi-3-medium' in m or 'phi-3-medium-4k' in m:
        aliases.extend(['phi-3-medium-4k', 'phi-3-medium', 'phi-3-medium-4k-instruct', 'phi3-medium'])
    if 'deepseek' in m:
        aliases.extend(['deepseek-moe-16b', 'deepseek_moe_16b', 'deepseek-moe-16b-chat', 'deepseek-moe', 'deepseek'])
    return list(set(aliases))

def _dataset_aliases(dataset):
    d = str(dataset).lower().replace('_', '-')
    aliases = [d, d.replace('-', '_')]
    if any(t in d for t in ['truth', 'truthful', 'truthfulqa']):
        aliases.extend(['truth', 'truthful', 'truthfulqa', 'truthful_qa'])
    elif any(s in d for s in ['science', 'scienceqa', 'sciq']):
        aliases.extend(['science', 'scienceqa', 'science_qa', 'sciq'])
    elif any(o in d for o in ['open', 'openbook', 'openbookqa']):
        aliases.extend(['open', 'openbook', 'openbookqa', 'openbook_qa'])
    return list(set(aliases))

def resolve_metric_candidates(chunk, candidate_keys, N):
    if not chunk or N <= 0: return np.full(N, np.nan, dtype=float)
    if isinstance(candidate_keys, str): candidate_keys = [candidate_keys]
    first_d = next((d for d in chunk if isinstance(d, dict)), {})
    for ck in candidate_keys:
        if ck in first_d or any(isinstance(d, dict) and ck in d for d in chunk[:N]):
            vals = [d.get(ck, np.nan) if isinstance(d, dict) else np.nan for d in chunk[:N]]
            return _to_fixed_array(vals, None, N)
    base_key = candidate_keys[0]
    alts = [k for k in first_d.keys() if k.startswith(f"{base_key}_") or base_key in k]
    if alts:
        alts.sort(key=lambda x: int(x.split('_')[-1]) if x.split('_')[-1].isdigit() else 999)
        vals = [d.get(alts[0], np.nan) if isinstance(d, dict) else np.nan for d in chunk[:N]]
        return _to_fixed_array(vals, None, N)
    return np.full(N, np.nan, dtype=float)

def load_run_data(run):
    with open(run['entropy_file'], 'rb') as f: raw_entropies = pkl.load(f)
    with open(run['label_file'], 'rb') as f: raw_labels = pkl.load(f)
    
    questions = []
    if run.get('question_file') and os.path.exists(run['question_file']):
        try:
            with open(run['question_file'], 'rb') as f: questions = pkl.load(f)
        except Exception:
            pass
            
    raw_labels_len = len(raw_labels)
    raw_ent_len = len(raw_entropies) if isinstance(raw_entropies, list) else 0
    
    if raw_ent_len >= int(2.5 * raw_labels_len) and raw_labels_len > 0:
        num_instances = raw_labels_len
        ours_entropies = raw_entropies[0 : num_instances]
        sdlg_entropies = raw_entropies[num_instances : 2*num_instances]
        kuhn_entropies = raw_entropies[2*num_instances : 3*num_instances]
    else:
        num_instances = min(raw_labels_len, raw_ent_len) if raw_ent_len > 0 else raw_labels_len
        ours_entropies = raw_entropies[0 : num_instances] if raw_ent_len >= num_instances else []
        sdlg_entropies = None
        kuhn_entropies = None

    m_aliases = _model_aliases(run.get('model', ''))
    d_aliases = _dataset_aliases(run.get('dataset', 'truthfulqa'))
    
    # 1. HYBRID FALLBACK: If SE chunks are missing (e.g. DATA_DIR is entropies3 or entropies_unified), lookup entropies2
    has_se = (ours_entropies and any(isinstance(d, dict) and 'normalised_semantic_entropy' in d for d in ours_entropies))
    if sdlg_entropies is None or kuhn_entropies is None or not has_se:
        candidate_e2_dirs = [
            '../experiments/result/entropies2',
            '../experiments/results/entropies2',
            'src/experiments/result/entropies2',
            'src/experiments/results/entropies2',
            './result/entropies2',
            '../../result/entropies2',
            '../experiments/result/entropies3',
            'src/experiments/result/entropies3',
            'result/entropies2',
            'result/entropies3',
            'entropies2',
            'entropies3',
            'src/temp',
            './src/temp',
            '../src/temp'
        ]
        for e2_dir in candidate_e2_dirs:
            if not os.path.exists(e2_dir): continue
            for fname in os.listdir(e2_dir):
                fl = fname.lower()
                if ('_entropies_' in fl or fl.startswith('sdlg_entropies_')) and any(ma in fl for ma in m_aliases) and any(da in fl for da in d_aliases):
                    e2_path = os.path.join(e2_dir, fname)
                    if os.path.abspath(e2_path) == os.path.abspath(run['entropy_file']):
                        continue
                    try:
                        with open(e2_path, 'rb') as fp: e2_raw = pkl.load(fp)
                        if isinstance(e2_raw, list) and len(e2_raw) >= int(2.5 * num_instances):
                            if sdlg_entropies is None: sdlg_entropies = e2_raw[num_instances : 2*num_instances]
                            if kuhn_entropies is None: kuhn_entropies = e2_raw[2*num_instances : 3*num_instances]
                            if not has_se:
                                ours_se_c = e2_raw[0:num_instances]
                                for idx_i in range(min(len(ours_entropies), len(ours_se_c))):
                                    if isinstance(ours_entropies[idx_i], dict) and isinstance(ours_se_c[idx_i], dict) and 'normalised_semantic_entropy' in ours_se_c[idx_i]:
                                        ours_entropies[idx_i]['normalised_semantic_entropy'] = ours_se_c[idx_i]['normalised_semantic_entropy']
                                has_se = True
                            break
                        elif isinstance(e2_raw, list) and len(e2_raw) >= num_instances and not has_se:
                            ours_se_c = e2_raw[0:num_instances]
                            if any(isinstance(d, dict) and 'normalised_semantic_entropy' in d for d in ours_se_c):
                                for idx_i in range(min(len(ours_entropies), len(ours_se_c))):
                                    if isinstance(ours_entropies[idx_i], dict) and isinstance(ours_se_c[idx_i], dict) and 'normalised_semantic_entropy' in ours_se_c[idx_i]:
                                        ours_entropies[idx_i]['normalised_semantic_entropy'] = ours_se_c[idx_i]['normalised_semantic_entropy']
                                has_se = True
                    except Exception:
                        pass
            if sdlg_entropies is not None and kuhn_entropies is not None and has_se:
                break

    labels = _to_fixed_array([safe_extract_label(l, EVAL_LABEL_KEY) for l in raw_labels[:num_instances]], None, num_instances)
    our_se = _to_fixed_array(ours_entropies, 'normalised_semantic_entropy', num_instances)
    sdlg_se = _to_fixed_array(sdlg_entropies, 'normalised_semantic_entropy', num_instances) if sdlg_entropies is not None else our_se
    kuhn_se = _to_fixed_array(kuhn_entropies, 'normalised_semantic_entropy', num_instances)
    q_list = questions[:num_instances] if len(questions) >= num_instances else (questions + [f"Q{i}" for i in range(len(questions), num_instances)])

    model_class, ka_depth, ke_depth = get_model_class_and_horizons(run['model'])
    ka_candidates = [f'entropy_mixture_top_{ka_depth}', f'entropy_mixture_{ka_depth-1}', f'entropy_mixture_depth_{ka_depth}', f'entropy_mixture_{ka_depth}', 'entropy_mixture_top_2', 'entropy_mixture_2', 'entropy_mixture']
    ke_candidates = [f'entropy_mixture_top_{ke_depth}', f'entropy_mixture_{ke_depth-1}', f'entropy_mixture_depth_{ke_depth}', f'entropy_mixture_{ke_depth}', 'entropy_mixture_top_16', 'entropy_mixture_16', 'entropy_mixture']
    h_final_calc = resolve_metric_candidates(ours_entropies, ['entropy_final', 'entropy_final_top_8'], num_instances)
    h_mix_ka_calc = resolve_metric_candidates(ours_entropies, ka_candidates, num_instances)
    h_mix_ke_calc = resolve_metric_candidates(ours_entropies, ke_candidates, num_instances)
    ep_diff_calc = h_final_calc - h_mix_ke_calc

    df = pd.DataFrame({
        'Question': q_list[:num_instances],
        'Correctness': labels,
        'Is_Correct': (pd.Series(labels) >= GLOBAL_CORR_THRESHOLD).astype(int),
        'Our_Weighted_SE': our_se,
        'entropy_final': h_final_calc,
        'entropy_mixture': h_mix_ka_calc,
        'difference_total': h_final_calc,
        'epistemic_diff': ep_diff_calc,
        'epistemic_diff_clipped': np.maximum(0.0, ep_diff_calc),
        'mutual_info': resolve_metric_candidates(ours_entropies, ['mutual_info_top_16', 'mutual_info_16', 'mutual_info_depth_16', 'mutual_info_15', 'mutual_info', 'mutual_info_top_8'], num_instances),
        'mutual_info_top_16': resolve_metric_candidates(ours_entropies, ['mutual_info_top_16', 'mutual_info_16', 'mutual_info_depth_16', 'mutual_info_15', 'mutual_info_top_8', 'mutual_info'], num_instances),
        'gated_mutual_info': resolve_metric_candidates(ours_entropies, ['gated_mutual_info_top_16', 'gated_mutual_info_top_8', 'gated_mutual_info', 'gated_mutual_info_top_24'], num_instances),
        'gated_mutual_info_depth_75': resolve_metric_candidates(ours_entropies, ['gated_mutual_info_depth_75', 'gated_mutual_info_75', 'gated_mutual_info_top_24', 'gated_mutual_info_top_21'], num_instances),
        'jsd': resolve_metric_candidates(ours_entropies, ['jsd_top_16', 'jsd_top_8', 'jsd'], num_instances),
        'kl': resolve_metric_candidates(ours_entropies, ['kl_top_16', 'kl_top_8', 'kl'], num_instances),
        'epkl': resolve_metric_candidates(ours_entropies, ['epkl_top_16', 'epkl_top_8', 'epkl'], num_instances),
        'tvd': resolve_metric_candidates(ours_entropies, ['tvd_top_16', 'tvd_top_8', 'tvd'], num_instances),
        'cv': resolve_metric_candidates(ours_entropies, ['cv_top_16', 'cv_top_8', 'cv'], num_instances),
        'bhattacharyya': resolve_metric_candidates(ours_entropies, ['bhattacharyya_top_16', 'bhattacharyya_top_8', 'bhattacharyya'], num_instances),
        'SDLG_Baseline_SE': sdlg_se,
        'SDLG_SE': sdlg_se,
        'Semantic_Entropy': sdlg_se,
        'Kuhn_Baseline_SE': kuhn_se,
        'Kuhn_SE': kuhn_se,
    })

    # External TokUR and ICE baselines
    candidate_tokur_roots = [
        os.path.join(DATA_DIR, '..', 'tokur', 'result') if DATA_DIR else '.',
        os.path.join(DATA_DIR, '..', 'tokur') if DATA_DIR else '.',
        os.path.join(DATA_DIR, '..', 'entropies_unified') if DATA_DIR else '.',
        './tokur/result', './tokur',
        '../experiments/tokur/result', '../experiments/tokur',
        'src/experiments/tokur/result', 'src/experiments/tokur',
        './result/tokur', 'results/tokur', 'result/tokur',
        'result/entropies_unified', './result/entropies_unified', 'entropies_unified',
        'entropies3', 'entropies2', '.'
    ]
    candidate_ice_roots = [
        os.path.join(DATA_DIR, '..', 'ice', 'result') if DATA_DIR else '.',
        os.path.join(DATA_DIR, '..', 'ice') if DATA_DIR else '.',
        './ice/result', './ice',
        '../experiments/ice/result', '../experiments/ice',
        'src/experiments/ice/result', 'src/experiments/ice',
        './result/ice', 'results/ice', 'result/ice',
        'entropies3', 'entropies2', '.'
    ]

    tokur_found = False
    for troot in candidate_tokur_roots:
        if tokur_found or not os.path.exists(troot): continue
        for root_dir, dirs, files in os.walk(troot):
            sorted_files = sorted(files, key=lambda x: (0 if 'entrop' in x.lower() else (1 if 'tokur' in x.lower() else 2)))
            for fname in sorted_files:
                fl = fname.lower()
                r_lower = root_dir.lower()
                if 'checkpoint' in fl or 'checkpoint' in r_lower:
                    continue
                if ('tokur' in fl or 'tokur' in r_lower) and any(ma in fl or ma in r_lower for ma in m_aliases) and any(da in fl or da in r_lower for da in d_aliases):
                    if fl.endswith('.pkl'):
                        tpath = os.path.join(root_dir, fname)
                        try:
                            with open(tpath, 'rb') as fp: tent = pkl.load(fp)
                            if isinstance(tent, list) and len(tent) > 0:
                                def _ext_tok(e, k1, k2):
                                    if not isinstance(e, dict): return np.nan
                                    sub = e.get('entropy_dict', e) if isinstance(e.get('entropy_dict'), dict) else e
                                    return sub.get(k1, sub.get(k2, np.nan))
                                tok_au = _to_fixed_array([_ext_tok(e, 'tokur_au', 'entropy_final') for e in tent], None, num_instances)
                                tok_mi = _to_fixed_array([_ext_tok(e, 'tokur_mi', 'mutual_info') for e in tent], None, num_instances)
                                if np.isfinite(tok_au).any() or np.isfinite(tok_mi).any():
                                    df['tokur_au'] = tok_au
                                    df['tokur_mi'] = tok_mi
                                    df['tokur_total'] = np.where(np.isfinite(tok_au) & np.isfinite(tok_mi), tok_au + tok_mi, np.nan)
                                    tokur_found = True
                                    break
                        except Exception: pass
            if tokur_found: break

    ice_found = False
    for iroot in candidate_ice_roots:
        if ice_found or not os.path.exists(iroot): continue
        for root_dir, dirs, files in os.walk(iroot):
            for fname in files:
                fl = fname.lower()
                r_lower = root_dir.lower()
                if ('ice' in fl or 'ice' in r_lower) and any(ma in fl or ma in r_lower for ma in m_aliases) and any(da in fl or da in r_lower for da in d_aliases):
                    if fl.endswith('.csv') or fl.endswith('.pkl'):
                        ipath = os.path.join(root_dir, fname)
                        try:
                            idf = pd.read_csv(ipath) if fl.endswith('.csv') else pd.DataFrame(pkl.load(open(ipath, 'rb')))
                            if 'aleatoric_uncertainty' in idf.columns:
                                df['ice_aleatoric'] = _to_fixed_array(idf['aleatoric_uncertainty'].values, None, num_instances)
                            if 'epistemic_uncertainty' in idf.columns:
                                df['ice_epistemic'] = _to_fixed_array(idf['epistemic_uncertainty'].values, None, num_instances)
                            if 'total_uncertainty' in idf.columns:
                                df['ice_total'] = _to_fixed_array(idf['total_uncertainty'].values, None, num_instances)
                            elif 'ice_aleatoric' in df.columns and 'ice_epistemic' in df.columns:
                                df['ice_total'] = df['ice_aleatoric'] + df['ice_epistemic']
                            if 'ice_aleatoric' in df.columns or 'ice_total' in df.columns or 'ice_epistemic' in df.columns:
                                ice_found = True
                                break
                        except Exception: pass
            if ice_found: break

    # Extract layerwise ensembling keys for sweep charts
    first_d = next((d for d in ours_entropies if isinstance(d, dict)), {})
    
    # 1. Map all entropy mixture depths (up to physical model depth <= 48)
    em_found_keys = [k for k in first_d.keys() if ('entropy_mixture' in k) and any(c.isdigit() for c in k)]
    for key in em_found_keys:
        df[key] = _to_fixed_array([d.get(key, 0.0) if isinstance(d, dict) else 0.0 for d in ours_entropies], None, num_instances)
        digits = ''.join(c for c in key.split('_')[-1] if c.isdigit())
        if digits and int(digits) <= 48:
            idx_k = int(digits) - 1 if int(digits) > 0 else 0
            df[f'entropy_mixture_{idx_k}'] = df[key]
            
    # 2. Map all mutual info depths (up to physical model depth <= 48)
    mi_found_keys = [k for k in first_d.keys() if ('mutual_info' in k and 'gated' not in k) and any(c.isdigit() for c in k)]
    for key in mi_found_keys:
        df[key] = _to_fixed_array([d.get(key, 0.0) if isinstance(d, dict) else 0.0 for d in ours_entropies], None, num_instances)
        digits = ''.join(c for c in key.split('_')[-1] if c.isdigit())
        if digits and int(digits) <= 48:
            idx_k = int(digits) - 1 if int(digits) > 0 else 0
            df[f'mutual_info_{idx_k}'] = df[key]

    # 3. Map all gated mutual info depths
    gmi_found_keys = [k for k in first_d.keys() if ('gated_mutual_info' in k) and any(c.isdigit() for c in k)]
    for key in gmi_found_keys:
        df[key] = _to_fixed_array([d.get(key, 0.0) if isinstance(d, dict) else 0.0 for d in ours_entropies], None, num_instances)
        digits = ''.join(c for c in key.split('_')[-1] if c.isdigit())
        if digits and int(digits) <= 48:
            idx_k = int(digits) - 1 if int(digits) > 0 else 0
            df[f'gated_mutual_info_{idx_k}'] = df[key]
            
    return df

def get_curves_and_metrics(df, uncertainty_col, threshold=GLOBAL_CORR_THRESHOLD):
    df['Is_Correct'] = (df['Correctness'] >= threshold).astype(int)
    u_raw = pd.to_numeric(df[uncertainty_col], errors='coerce')
    c_raw = pd.to_numeric(df['Correctness'], errors='coerce')
    valid_mask = u_raw.notna() & c_raw.notna() & np.isfinite(u_raw) & np.isfinite(c_raw)
    df_clean = df[valid_mask].copy()
    
    if len(df_clean) < 2:
        return [0.0, 1.0], [0.0, 1.0], np.nan, np.nan, np.linspace(0.01, 1.0, 100), [], [], 0.0
        
    correct = df_clean['Is_Correct'].values
    u_val = pd.to_numeric(df_clean[uncertainty_col], errors='coerce').values
    N = len(correct)
    a0 = correct.mean()
    
    try:
        if len(np.unique(correct)) > 1:
            fpr, tpr, _ = roc_curve(1 - correct, u_val)
            auroc = roc_auc_score(1 - correct, u_val)
        else:
            fpr, tpr, auroc = [0.0, 1.0], [0.0, 1.0], 0.500
    except Exception:
        fpr, tpr, auroc = [0.0, 1.0], [0.0, 1.0], 0.500

    try:
        correct_sorted = df_clean.sort_values(by=uncertainty_col, ascending=True)['Is_Correct'].values
        coverages = np.linspace(0.01, 1.0, 100)
        accuracies_method = [correct_sorted[:max(1, int(N * cov))].mean() for cov in coverages]
        accuracies_oracle = [min(max(1, int(N * cov)), correct.sum()) / max(1, int(N * cov)) for cov in coverages]
        area_method = np.trapz(accuracies_method, coverages)
        area_oracle = np.trapz(accuracies_oracle, coverages)
        area_baseline = a0
        prr = (area_method - area_baseline) / (area_oracle - area_baseline + 1e-10)
    except Exception:
        prr = 0.0
        coverages, accuracies_method, accuracies_oracle = np.linspace(0.01, 1.0, 100), [], []
        
    return fpr, tpr, auroc, prr, coverages, accuracies_method, accuracies_oracle, a0

def calculate_auroc_and_prr(df, uncertainty_col, threshold=GLOBAL_CORR_THRESHOLD):
    fpr, tpr, auroc, prr, _, _, _, _ = get_curves_and_metrics(df, uncertainty_col, threshold)
    return auroc, prr

def bold_max_grouped(df):
    if df.empty:
        return df
    level0_groups = []
    for col in df.columns:
        if col[0] not in level0_groups:
            level0_groups.append(col[0])
    boundary_col_indices = []
    for grp in level0_groups[:-1]:
        grp_indices = [i for i, col in enumerate(df.columns) if col[0] == grp]
        if grp_indices:
            boundary_col_indices.append(grp_indices[-1])
    def _highlight_max_and_borders(row):
        styles = [''] * len(row)
        groups = set(col[0] for col in df.columns if col[0] != 'Meta')
        for grp in groups:
            grp_cols = [idx for idx, col in enumerate(df.columns) if col[0] == grp]
            vals = [row.iloc[idx] for idx in grp_cols]
            numeric_vals = [(idx, v) for idx, v in zip(grp_cols, vals) if pd.notna(v) and isinstance(v, (int, float, np.number))]
            if numeric_vals:
                max_idx, max_v = max(numeric_vals, key=lambda x: x[1])
                for idx, v in numeric_vals:
                    if abs(v - max_v) < 1e-6:
                        styles[idx] = 'font-weight: bold; background-color: #d4edda;'
        for b_idx in boundary_col_indices:
            styles[b_idx] += ' border-right: 2.5px solid #444444;'
        return styles
    format_dict = {}
    for col in df.columns:
        if col[0] != 'Meta':
            format_dict[col] = '{:.4f}'
    styler = df.style.apply(_highlight_max_and_borders, axis=1).format(format_dict, na_rep='-')
    table_styles = [
        {'selector': 'th', 'props': [('text-align', 'center'), ('font-size', '10.5pt')]},
        {'selector': 'td', 'props': [('text-align', 'center'), ('padding', '5px 8px')]}
    ]
    for b_idx in boundary_col_indices:
        table_styles.append({
            'selector': f'th.col{b_idx}',
            'props': [('border-right', '2.5px solid #444444 !important')]
        })
    styler = styler.set_table_styles(table_styles, overwrite=False)
    return styler


## 2. Full-Width BLEURT Threshold Sweep Plots: $\tau \in [-0.70, +0.50]$
Evaluates AUROC stability across 25 threshold steps for Difference Decomposition, Premature Ensembling, Gated MI, TokUR, ICE, and Semantic Entropy baselines.

In [ ]:
# Metric Catalog & High-Contrast Visual Styling for Threshold Sweeps
# Contains all baselines and focuses strictly on our core Difference Decomposition metrics
metric_specs = [
    # --- Group 1: Ours - Difference Decomposition (Core Proposed Metrics) ---
    ('difference_total',         'Ours: Difference Total (AU+EU)', '#15803d', '-',  'o', 2.8, 14),
    ('entropy_mixture',          'Ours: H_mix (Class Ka*)',        '#16a34a', '-',  's', 2.6, 13),
    ('epistemic_diff',           'Ours: Difference EU (Class Ke*)','#059669', '-',  'D', 2.6, 13),
    
    # --- Group 2: Single-Layer Baseline (Dotted Line) ---
    ('entropy_final',            'Baseline: H_final (Single-Layer)', '#334155', ':', 'v', 2.2, 11),
    
    # --- Group 3: Semantic Entropy Baselines (Dashed Lines) ---
    ('Kuhn_SE',                  'Kuhn SE Baseline',               '#7c3aed', '--', 'x', 2.2, 10),
    ('SDLG_SE',                  'SDLG SE (Unweighted)',           '#d97706', '--', 's', 2.2, 10),
    
    # --- Group 4: TokUR Baselines (Dash-Dot Lines) ---
    ('tokur_total',              'TokUR Total',                    '#0f766e', '-.', 'X', 2.2, 9),
    ('tokur_au',                 'TokUR AU',                       '#0d9488', '-.', '*', 2.2, 9),
    ('tokur_mi',                 'TokUR EU (MI)',                  '#06b6d4', '-.', 'P', 2.2, 9),
    
    # --- Group 5: ICE Baselines (Dash-Dot Lines) ---
    ('ice_total',                'ICE Total',                      '#be185d', '-.', '+', 2.0, 8),
    ('ice_aleatoric',            'ICE Aleatoric',                  '#b45309', '-.', 'd', 2.0, 8),
    ('ice_epistemic',            'ICE Epistemic',                  '#c2410c', '-.', 'h', 2.0, 8),
    
    # --- Group 6: Disagreement EU Baselines (Dotted Lines) ---
    ('jsd',                      'JSD',                            '#64748b', ':',  '.', 1.6, 7),
    ('kl',                       'KL Divergence',                  '#94a3b8', ':',  '.', 1.6, 7),
    ('epkl',                     'EPKL',                           '#475569', ':',  '1', 1.6, 7),
    ('tvd',                      'TVD',                            '#9ca3af', ':',  '.', 1.6, 7),
]

candidate_thresholds = np.linspace(-0.70, 0.50, 25)
sweep_results = []

if not runs:
    print("[WARNING] No registered runs found in data directories. Please check candidate folders.")

for idx, run in enumerate(runs):
    df = load_run_data(run)
    if df is None or (isinstance(df, pd.DataFrame) and df.empty):
        print(f"[SKIP] Could not load valid data for run: {run['name']}")
        continue
    if isinstance(df, tuple):
        df = df[0]
        
    m_name = run['model']
    d_name = run['dataset']
    scores_arr = df['Correctness'].values
    m_cls, ka_val, ke_val = get_model_class_and_horizons(m_name)
    
    if 'difference_total' not in df.columns:
        df['difference_total'] = df['entropy_final'] if 'entropy_final' in df.columns else (df.get('entropy_mixture', 0) + df.get('epistemic_diff', 0))
    
    fig, ax = plt.subplots(figsize=(11.5, 6.2))
    plotted_count = 0
    model_sweep_record = {'model': m_name, 'dataset': d_name, 'class': m_cls}
    
    for col_k, display_lbl, color_hex, line_style, marker_sym, line_w, z_idx in metric_specs:
        if col_k in df.columns and df[col_k].notna().any():
            u_vals = df[col_k].values
            valid_mask = pd.notna(u_vals) & pd.notna(scores_arr)
            
            aurocs = []
            for th in candidate_thresholds:
                y_err = (scores_arr[valid_mask] < th).astype(int)
                if len(np.unique(y_err)) >= 2:
                    try:
                        auc = roc_auc_score(y_err, np.array(u_vals[valid_mask], dtype=float))
                        aurocs.append(auc)
                    except Exception:
                        aurocs.append(np.nan)
                else:
                    aurocs.append(np.nan)
                    
            if any(pd.notna(a) for a in aurocs):
                ax.plot(
                    candidate_thresholds, 
                    aurocs, 
                    label=display_lbl, 
                    color=color_hex, 
                    linestyle=line_style, 
                    marker=marker_sym, 
                    markersize=4.5, 
                    linewidth=line_w, 
                    zorder=z_idx,
                    alpha=0.92
                )
                plotted_count += 1
                model_sweep_record[("AUROC @ 0.00", display_lbl)] = aurocs[np.argmin(np.abs(candidate_thresholds - 0.0))]
                model_sweep_record[("Mean AUROC [-0.7, 0.5]", display_lbl)] = np.nanmean(aurocs)
                
    if plotted_count > 0:
        ax.axvline(GLOBAL_CORR_THRESHOLD, color='#e11d48', linestyle='--', label=f'Standard Target ({GLOBAL_CORR_THRESHOLD:+.2f})', linewidth=1.8, zorder=15)
        ax.axhline(0.50, color='gray', linestyle=':', label='Random Guess (0.50)', linewidth=1.3)
        
        ax.set_title(f"BLEURT Correctness Threshold Sweep (\u03c4 \u2208 [-0.70, +0.50]) \u2014 {d_name}: {m_name} ({m_cls} Ka*={ka_val}, Ke*={ke_val})", fontsize=13.5, fontweight='bold', pad=10)
        ax.set_xlabel("Correctness Threshold (BLEURT Score)", fontsize=12, labelpad=8)
        ax.set_ylabel("Error Detection AUROC Score", fontsize=12, labelpad=8)
        ax.set_xlim(-0.72, 0.52)
        ax.set_ylim(0.35, 0.88)
        ax.grid(True, linestyle='--', alpha=0.5)
        ax.legend(bbox_to_anchor=(1.01, 1.0), loc='upper left', fontsize=9.2, frameon=True, shadow=True, borderaxespad=0)
        
        plt.tight_layout()
        plt.show()
        sweep_results.append(model_sweep_record)
    else:
        print(f"[WARNING] No plottable UQ metrics found for {run['name']}.")
        plt.close(fig)


## 3. Consolidated AUROC Benchmark Across Key BLEURT Thresholds
Hierarchical grouped comparison of AUROC performance at the standard threshold ($\tau = 0.0$) and averaged across the entire sweep spectrum ($\tau \in [-0.70, +0.50]$).

In [ ]:
def bold_max_grouped(df):
    if df.empty: return df
    level0_groups = []
    for col in df.columns:
        if col[0] not in level0_groups:
            level0_groups.append(col[0])
    boundary_col_indices = []
    for grp in level0_groups[:-1]:
        grp_indices = [i for i, col in enumerate(df.columns) if col[0] == grp]
        if grp_indices:
            boundary_col_indices.append(grp_indices[-1])
    def _highlight_max_and_borders(row):
        styles = [''] * len(row)
        groups = set(col[0] for col in df.columns if col[0] != 'Meta')
        for grp in groups:
            grp_cols = [idx for idx, col in enumerate(df.columns) if col[0] == grp]
            vals = [row.iloc[idx] for idx in grp_cols]
            numeric_vals = [(idx, v) for idx, v in zip(grp_cols, vals) if pd.notna(v) and isinstance(v, (int, float, np.number))]
            if numeric_vals:
                max_idx, max_v = max(numeric_vals, key=lambda x: x[1])
                for idx, v in numeric_vals:
                    if abs(v - max_v) < 1e-6:
                        styles[idx] = 'font-weight: bold; background-color: #d4edda;'
        for b_idx in boundary_col_indices:
            styles[b_idx] += ' border-right: 2.5px solid #444444;'
        return styles
    format_dict = {}
    for col in df.columns:
        if col[0] != 'Meta':
            format_dict[col] = '{:.4f}'
    styler = df.style.apply(_highlight_max_and_borders, axis=1).format(format_dict, na_rep='-')
    table_styles = [
        {'selector': 'th', 'props': [('text-align', 'center'), ('font-size', '10.5pt')]},
        {'selector': 'td', 'props': [('text-align', 'center'), ('padding', '5px 8px')]}
    ]
    for b_idx in boundary_col_indices:
        table_styles.extend([
            {'selector': f'th.col{b_idx}', 'props': [('border-right', '2.5px solid #444444 !important')]},
            {'selector': f'th.col_heading.col{b_idx}', 'props': [('border-right', '2.5px solid #444444 !important')]},
            {'selector': f'th.col_heading.level0.col{b_idx}', 'props': [('border-right', '2.5px solid #444444 !important')]},
            {'selector': f'th.col_heading.level1.col{b_idx}', 'props': [('border-right', '2.5px solid #444444 !important')]},
            {'selector': f'td.col{b_idx}', 'props': [('border-right', '2.5px solid #444444 !important')]}
        ])
    styler = styler.set_table_styles(table_styles, overwrite=False)
    return styler

candidate_thresholds = np.linspace(-0.70, 0.50, 25)

# Grouped Metric Specifications for Threshold Benchmarks
# Balanced: Core Difference Decomposition metrics + Complete literature baselines
grouped_sweep_metrics = [
    # Total Uncertainty (TU)
    ("Total Uncertainty (TU)", "Difference Total (Ours)", "difference_total"),
    ("Total Uncertainty (TU)", "H_final (Single-Layer)", "entropy_final"),
    ("Total Uncertainty (TU)", "TokUR Total", "tokur_total"),
    ("Total Uncertainty (TU)", "ICE Total", "ice_total"),
    
    # Aleatoric Uncertainty (AU)
    ("Aleatoric Uncertainty (AU)", "H_mix (Ours Class Ka*)", "entropy_mixture"),
    ("Aleatoric Uncertainty (AU)", "TokUR AU", "tokur_au"),
    ("Aleatoric Uncertainty (AU)", "ICE AU", "ice_aleatoric"),
    ("Aleatoric Uncertainty (AU)", "SDLG SE (Unweighted)", "SDLG_SE"),
    ("Aleatoric Uncertainty (AU)", "Kuhn SE Baseline", "Kuhn_SE"),
    
    # Epistemic Uncertainty (EU)
    ("Epistemic Uncertainty (EU)", "Difference EU (Ours Class Ke*)", "epistemic_diff"),
    ("Epistemic Uncertainty (EU)", "TokUR EU", "tokur_mi"),
    ("Epistemic Uncertainty (EU)", "ICE EU", "ice_epistemic"),
    ("Epistemic Uncertainty (EU)", "JSD", "jsd"),
    ("Epistemic Uncertainty (EU)", "KL", "kl"),
    ("Epistemic Uncertainty (EU)", "EPKL", "epkl"),
    ("Epistemic Uncertainty (EU)", "TVD", "tvd"),
]

table_rows_std = []
table_rows_mean = []

for run in runs:
    df = load_run_data(run)
    if df is None or (isinstance(df, pd.DataFrame) and df.empty): continue
    if isinstance(df, tuple): df = df[0]
    
    m_name = run['model']
    d_name = run['dataset']
    scores_arr = df['Correctness'].values
    m_cls, ka_val, ke_val = get_model_class_and_horizons(m_name)
    acc_val = f"{df['Is_Correct'].mean() * 100:.1f}%" if 'Is_Correct' in df.columns else f"{(scores_arr >= GLOBAL_CORR_THRESHOLD).mean() * 100:.1f}%"
    
    if 'difference_total' not in df.columns:
        df['difference_total'] = df['entropy_final'] if 'entropy_final' in df.columns else (df.get('entropy_mixture', 0) + df.get('epistemic_diff', 0))
    
    r_std = {
        ("Meta", "Dataset"): d_name,
        ("Meta", "Model"): m_name,
        ("Meta", "Class"): m_cls,
        ("Meta", "Ka*"): ka_val,
        ("Meta", "Ke*"): ke_val,
        ("Meta", "Acc"): acc_val
    }
    r_mean = {
        ("Meta", "Dataset"): d_name,
        ("Meta", "Model"): m_name,
        ("Meta", "Class"): m_cls,
        ("Meta", "Ka*"): ka_val,
        ("Meta", "Ke*"): ke_val,
        ("Meta", "Acc"): acc_val
    }
    
    for grp, lbl, col_k in grouped_sweep_metrics:
        col_tuple = (grp, lbl)
        if col_k in df.columns and df[col_k].notna().any():
            u_vals = df[col_k].values
            valid_mask = pd.notna(u_vals) & pd.notna(scores_arr)
            
            # Standard Target Threshold (0.00)
            y_err_std = (scores_arr[valid_mask] < GLOBAL_CORR_THRESHOLD).astype(int)
            if len(np.unique(y_err_std)) >= 2:
                try: r_std[col_tuple] = roc_auc_score(y_err_std, np.array(u_vals[valid_mask], dtype=float))
                except Exception: r_std[col_tuple] = np.nan
            else:
                r_std[col_tuple] = np.nan
                
            # Mean AUROC across full sweep [-0.70, +0.50]
            all_auc = []
            for th in candidate_thresholds:
                y_err_th = (scores_arr[valid_mask] < th).astype(int)
                if len(np.unique(y_err_th)) >= 2:
                    try: all_auc.append(roc_auc_score(y_err_th, np.array(u_vals[valid_mask], dtype=float)))
                    except Exception: pass
            r_mean[col_tuple] = np.nanmean(all_auc) if all_auc else np.nan
        else:
            r_std[col_tuple] = np.nan
            r_mean[col_tuple] = np.nan
            
    table_rows_std.append(r_std)
    table_rows_mean.append(r_mean)

df_tstd = pd.DataFrame(table_rows_std)
if not df_tstd.empty: df_tstd.columns = pd.MultiIndex.from_tuples(df_tstd.columns)

df_tmean = pd.DataFrame(table_rows_mean)
if not df_tmean.empty: df_tmean.columns = pd.MultiIndex.from_tuples(df_tmean.columns)

print("=" * 125)
print(f"TABLE 1: AUROC BENCHMARK AT STANDARD TARGET THRESHOLD (BLEURT >= {GLOBAL_CORR_THRESHOLD})")
print("=" * 125)
if not df_tstd.empty: display(bold_max_grouped(df_tstd))
else: print("No benchmark data available to display in Table 1.")

print("\n" + "=" * 125)
print("TABLE 2: MEAN AUROC ACROSS FULL BLEURT THRESHOLD SWEEP [-0.70, +0.50]")
print("=" * 125)
if not df_tmean.empty: display(bold_max_grouped(df_tmean))
else: print("No benchmark data available to display in Table 2.")
